# ReCAHS — Honest Deployment and Efficiency Analysis

Bu notebook temsili `seed=2026` modeli üzerinde deployment maliyetini ölçer. Accuracy sonucu üretmez; tamamlanmış beş-seed sonuçlarını değiştirmez.

Karşılaştırılan senaryolar:
1. **Unpruned baseline:** tek tam model.
2. **Soft-masked dynamic:** tek tam model + üç maske; parametre/FLOP kazancı yoktur.
3. **Static structural:** fiziksel olarak küçültülmüş tek model.
4. **Dynamic structural:** trend/seasonal/residual için üç küçültülmüş model; istek başına yalnızca biri aktiftir fakat üçünün de saklanması gerekir.

Ölçümler: parametreler, checkpoint boyutu, THOP FLOPs, randomized/interleaved GPU ve CPU latency, STL detector latency, uçtan uca latency ve 1.000/1.000.000 tahmin projeksiyonu.

In [ ]:
# 1) AYARLAR — seed=2026 temsili deployment modeli
from pathlib import Path

REPRESENTATIVE_SEED = 2026
TSLIB_DIR = Path('/content/Time-Series-Library')
TSLIB_COMMIT = '4e938a1767106324dd753b2a44832bf870a0252e'
GPU_REPEATS = 40
GPU_WARMUP = 8
CPU_REPEATS = 15
CPU_WARMUP = 3
STL_WINDOWS = 200
BENCHMARK_SEED = 2026

print('Representative seed:', REPRESENTATIVE_SEED)

In [ ]:
# 2) DRIVE, TSLIB, BAĞIMLILIKLAR VE VERİ
from google.colab import drive
import copy, gc, hashlib, importlib, json, math, os, random, shutil, subprocess, sys, time
from argparse import Namespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

primary_mount = Path('/content/drive')
if (primary_mount / 'MyDrive').is_dir():
    drive_root = primary_mount
else:
    try:
        drive.mount(str(primary_mount)); drive_root = primary_mount
    except ValueError as error:
        if 'already contain files' not in str(error): raise
        drive_root = Path('/content/gdrive')
        if not (drive_root / 'MyDrive').is_dir(): drive.mount(str(drive_root))

PROJECT_DIR = drive_root / 'MyDrive' / 'BIL401_Regime_Head_Pruning'
MULTISEED_DIR = PROJECT_DIR / 'multiseed' / 'ETTh1'
PRUNING_DIR = MULTISEED_DIR / 'pruning'
SEED_PRUNING_DIR = PRUNING_DIR / f'seed_{REPRESENTATIVE_SEED}'
EFFICIENCY_DIR = PRUNING_DIR / 'deployment_efficiency'
STRUCTURAL_CKPT_DIR = EFFICIENCY_DIR / 'structural_checkpoints'
for directory in [EFFICIENCY_DIR, STRUCTURAL_CKPT_DIR]: directory.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    print('$', ' '.join(map(str, command)))
    result=subprocess.run(list(map(str,command)),cwd=str(cwd) if cwd else None,text=True,capture_output=True)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip())
    if result.returncode != 0: raise RuntimeError(command)
    return result

if not (TSLIB_DIR/'.git').exists(): run(['git','clone','https://github.com/thuml/Time-Series-Library.git',TSLIB_DIR])
run(['git','fetch','origin'],cwd=TSLIB_DIR); run(['git','checkout','--force',TSLIB_COMMIT],cwd=TSLIB_DIR)
packages=['patool','sktime','scikit-base','einops','reformer-pytorch','local-attention','hyper-connections','axial-positional-embedding','product-key-memory','colt5-attention','thop']
run([sys.executable,'-m','pip','install','-q','--no-deps',*packages])
for module in ['einops','reformer_pytorch','thop']: importlib.import_module(module)

drive_data=PROJECT_DIR/'data'/'ETTh1.csv'; assert drive_data.exists(), drive_data
tslib_data=TSLIB_DIR/'dataset'/'ETDataset'/'ETT-small'/'ETTh1.csv'; tslib_data.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(drive_data,tslib_data)
if str(TSLIB_DIR) not in sys.path: sys.path.insert(0,str(TSLIB_DIR))
os.chdir(TSLIB_DIR)
device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Project:',PROJECT_DIR); print('Device:',device,torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3) MODEL, CHECKPOINT, MASKELER VE SAMPLE INPUT
from torch.utils.data import DataLoader
from data_provider.data_loader import Dataset_ETT_hour
from models.PatchTST import Model as PatchTSTModel

args=Namespace(task_name='long_term_forecast',data='ETTh1',root_path=str(tslib_data.parent)+'/',data_path='ETTh1.csv',features='M',target='OT',freq='h',embed='timeF',seasonal_patterns='Monthly',seq_len=336,label_len=48,pred_len=96,enc_in=7,dec_in=7,c_out=7,d_model=128,n_heads=8,e_layers=3,d_layers=1,d_ff=256,factor=3,dropout=0.1,activation='gelu',augmentation_ratio=0.0,batch_size=32,num_workers=0)

checkpoint_candidates=list((MULTISEED_DIR/f'seed_{REPRESENTATIVE_SEED}'/'checkpoints').glob('**/checkpoint.pth'))
assert len(checkpoint_candidates)==1, checkpoint_candidates
BASELINE_CHECKPOINT=checkpoint_candidates[0]
baseline_state=torch.load(BASELINE_CHECKPOINT,map_location='cpu',weights_only=True)

def load_fresh_model(target_device=device):
    model=PatchTSTModel(args); model.load_state_dict(baseline_state); model.to(target_device); model.eval(); return model

def load_mask(path):
    frame=pd.read_csv(path); values=frame.to_numpy(dtype=np.float32)
    assert values.shape==(3,8), (path,values.shape)
    mask=torch.tensor(values); assert int(mask.sum())==18, (path,int(mask.sum()))
    return mask

static_mask=load_mask(SEED_PRUNING_DIR/'static_keep_mask.csv')
joint_masks={regime:load_mask(SEED_PRUNING_DIR/f'{regime}_joint_keep_mask.csv') for regime in ['trend','seasonal','residual']}

test_data=Dataset_ETT_hour(args=args,root_path=args.root_path,flag='test',size=[336,48,96],features='M',data_path='ETTh1.csv',target='OT',timeenc=1,freq='h')
test_loader=DataLoader(test_data,batch_size=32,shuffle=False,num_workers=0,drop_last=False)
sample=next(iter(test_loader))
sample_gpu=tuple(t.float().to(device) for t in sample)
sample1_gpu=tuple(t[:1] for t in sample_gpu)
print('Checkpoint:',BASELINE_CHECKPOINT); print('Static active:',int(static_mask.sum())); print('Dynamic active:',{k:int(v.sum()) for k,v in joint_masks.items()})

In [ ]:
# 4) STRUCTURAL PRUNING VE SOFT-MASK EŞDEĞERLİK KONTROLÜ
def structurally_prune_attention_layer(attention_layer, keep_heads):
    dev=attention_layer.query_projection.weight.device; dtype=attention_layer.query_projection.weight.dtype
    d_model=attention_layer.query_projection.in_features; old_heads=attention_layer.n_heads
    d_keys=attention_layer.query_projection.out_features//old_heads; d_values=attention_layer.value_projection.out_features//old_heads
    new_heads=len(keep_heads)
    def slice_rows(linear,d_per_head):
        new=nn.Linear(linear.in_features,d_per_head*new_heads,bias=linear.bias is not None).to(dev,dtype)
        new.weight.data=torch.cat([linear.weight.data[h*d_per_head:(h+1)*d_per_head,:] for h in keep_heads],dim=0).clone()
        if linear.bias is not None: new.bias.data=torch.cat([linear.bias.data[h*d_per_head:(h+1)*d_per_head] for h in keep_heads]).clone()
        return new
    new_q=slice_rows(attention_layer.query_projection,d_keys); new_k=slice_rows(attention_layer.key_projection,d_keys); new_v=slice_rows(attention_layer.value_projection,d_values)
    new_out=nn.Linear(d_values*new_heads,d_model,bias=attention_layer.out_projection.bias is not None).to(dev,dtype)
    new_out.weight.data=torch.cat([attention_layer.out_projection.weight.data[:,h*d_values:(h+1)*d_values] for h in keep_heads],dim=1).clone()
    if attention_layer.out_projection.bias is not None: new_out.bias.data=attention_layer.out_projection.bias.data.clone()
    attention_layer.query_projection=new_q; attention_layer.key_projection=new_k; attention_layer.value_projection=new_v; attention_layer.out_projection=new_out; attention_layer.n_heads=new_heads

def mask_to_keep(mask): return [[h for h in range(8) if mask[layer,h].item()==1] for layer in range(3)]
def build_structural(mask,target_device=device):
    model=load_fresh_model(target_device)
    for layer,keep in enumerate(mask_to_keep(mask)): structurally_prune_attention_layer(model.encoder.attn_layers[layer].attention,keep)
    model.eval(); return model

baseline_model=load_fresh_model(device)
models={'baseline':baseline_model,'static_structural':build_structural(static_mask)}
for regime,mask in joint_masks.items(): models[f'dynamic_{regime}_structural']=build_structural(mask)

class SoftMaskController:
    def __init__(self,model): self.model=model; self.mask=None
    def install(self):
        for layer_idx,encoder_layer in enumerate(self.model.encoder.attn_layers):
            a=encoder_layer.attention
            def make_forward(layer_idx,a):
                def forward(q0,k0,v0,attn_mask,tau=None,delta=None):
                    B,L,_=q0.shape; _,S,_=k0.shape; H=a.n_heads
                    q=a.query_projection(q0).view(B,L,H,-1); k=a.key_projection(k0).view(B,S,H,-1); v=a.value_projection(v0).view(B,S,H,-1)
                    out,attn=a.inner_attention(q,k,v,attn_mask,tau=tau,delta=delta); out=out*self.mask[layer_idx].to(out.device).view(1,1,H,1)
                    return a.out_projection(out.reshape(B,L,-1)),attn
                return forward
            a.forward=make_forward(layer_idx,a)
    def set_mask(self,mask): self.mask=mask.clone().float()

reference=load_fresh_model(device); controller=SoftMaskController(reference); controller.install()
checks={'static_structural':static_mask,**{f'dynamic_{r}_structural':m for r,m in joint_masks.items()}}
correctness=[]
with torch.no_grad():
    for name,mask in checks.items():
        controller.set_mask(mask); soft=reference(*sample1_gpu); hard=models[name](*sample1_gpu); diff=float((soft-hard).abs().max())
        correctness.append({'setting':name,'max_abs_difference':diff}); assert diff<1e-3,(name,diff)
correctness_df=pd.DataFrame(correctness); display(correctness_df)
print('Tüm structural modeller soft-mask çıktılarıyla eşdeğer.')

In [ ]:
# 5) PARAMETRE, FLOP VE CHECKPOINT DEPOLAMA MALİYETİ
from thop import profile

def count_parameters(model): return sum(p.numel() for p in model.parameters())
parameter_counts={name:count_parameters(model) for name,model in models.items()}
flop_counts={}
for name,model in models.items():
    clone=copy.deepcopy(model); flops,_=profile(clone,inputs=sample1_gpu,verbose=False); flop_counts[name]=float(flops); del clone

structural_files={}
for name,model in models.items():
    path=STRUCTURAL_CKPT_DIR/f'{name}_seed{REPRESENTATIVE_SEED}.pth'; torch.save(model.state_dict(),path); structural_files[name]=path
checkpoint_bytes={name:path.stat().st_size for name,path in structural_files.items()}

five_seed_storage=[]
for seed in [7,42,1234,2026,3407]:
    paths=list((MULTISEED_DIR/f'seed_{seed}'/'checkpoints').glob('**/checkpoint.pth')); assert len(paths)==1
    five_seed_storage.append({'seed':seed,'checkpoint_bytes':paths[0].stat().st_size,'checkpoint_mib':paths[0].stat().st_size/1024**2})
five_seed_storage_df=pd.DataFrame(five_seed_storage); display(five_seed_storage_df)

metric_rows=[]
for name in models:
    metric_rows.append({'setting':name,'parameters':parameter_counts[name],'flops_thop':flop_counts[name],'checkpoint_bytes':checkpoint_bytes[name],'checkpoint_mib':checkpoint_bytes[name]/1024**2})
model_metrics_df=pd.DataFrame(metric_rows)
base_params=parameter_counts['baseline']; base_flops=flop_counts['baseline']
model_metrics_df['parameters_vs_baseline_percent']=(model_metrics_df.parameters/base_params-1)*100
model_metrics_df['flops_vs_baseline_percent']=(model_metrics_df.flops_thop/base_flops-1)*100
display(model_metrics_df)

In [ ]:
# 6) RANDOMIZED / INTERLEAVED GPU VE CPU LATENCY
def interleaved_benchmark(model_map,inputs,target_device,warmup,repeats,seed,batch_label):
    sx,sxm,sy,sym=inputs; rng=random.Random(seed); names=list(model_map); records=[]
    with torch.no_grad():
        for _ in range(warmup):
            order=names.copy(); rng.shuffle(order)
            for name in order: _=model_map[name](sx,sxm,sy,sym)
        if target_device.type=='cuda': torch.cuda.synchronize()
        for round_id in range(repeats):
            order=names.copy(); rng.shuffle(order)
            for position,name in enumerate(order):
                if target_device.type=='cuda': torch.cuda.synchronize()
                start=time.perf_counter(); _=model_map[name](sx,sxm,sy,sym)
                if target_device.type=='cuda': torch.cuda.synchronize()
                records.append({'device':target_device.type,'batch':batch_label,'round':round_id,'position':position,'setting':name,'latency_ms':(time.perf_counter()-start)*1000})
    raw=pd.DataFrame(records)
    summary=raw.groupby(['device','batch','setting']).latency_ms.agg(['mean','std','median',lambda x:np.percentile(x,95)]).reset_index()
    summary.columns=['device','batch','setting','mean_ms','std_ms','p50_ms','p95_ms']
    summary['batch_size']=sx.shape[0]; summary['mean_ms_per_sample']=summary.mean_ms/summary.batch_size
    return raw,summary

gpu_raw1,gpu_summary1=interleaved_benchmark(models,sample1_gpu,device,GPU_WARMUP,GPU_REPEATS,BENCHMARK_SEED,'batch1')
gpu_raw32,gpu_summary32=interleaved_benchmark(models,sample_gpu,device,GPU_WARMUP,GPU_REPEATS,BENCHMARK_SEED+1,'batch32')

cpu_models={name:copy.deepcopy(model).to('cpu').eval() for name,model in models.items()}
sample1_cpu=tuple(t.cpu() for t in sample1_gpu)
cpu_raw1,cpu_summary1=interleaved_benchmark(cpu_models,sample1_cpu,torch.device('cpu'),CPU_WARMUP,CPU_REPEATS,BENCHMARK_SEED,'batch1')
latency_raw_df=pd.concat([gpu_raw1,gpu_raw32,cpu_raw1],ignore_index=True)
latency_summary_df=pd.concat([gpu_summary1,gpu_summary32,cpu_summary1],ignore_index=True)
display(latency_summary_df)
del cpu_models; gc.collect()

In [ ]:
# 7) STL DETECTOR, DYNAMIC EXPECTED COST VE 1K / 1M PROJEKSİYONU
from statsmodels.tsa.seasonal import STL
raw_df=pd.read_csv(drive_data); train_size=12*30*24; val_size=4*30*24; seq_len=336
segment=raw_df.iloc[train_size-seq_len:train_size+val_size].reset_index(drop=True); target=segment.OT.to_numpy(dtype=float)
rng=random.Random(BENCHMARK_SEED); ids=sorted(rng.sample(range(len(target)-seq_len+1),STL_WINDOWS)); stl_times=[]
for window_id in ids:
    start=time.perf_counter(); _=STL(target[window_id:window_id+seq_len],period=24,robust=True).fit(); stl_times.append((time.perf_counter()-start)*1000)
stl_times=np.array(stl_times); stl_mean=float(stl_times.mean()); stl_std=float(stl_times.std(ddof=1))

test_labels=pd.read_csv(PROJECT_DIR/'regime_detection'/'etth1_test_regimes_ot_seq336.csv')
frequency=test_labels.regime.value_counts(normalize=True).to_dict(); print('Test regime frequency:',frequency)
def latency_value(setting,batch,dev='cuda'):
    return float(latency_summary_df.query('device==@dev and batch==@batch and setting==@setting').mean_ms.iloc[0])
dynamic_names={r:f'dynamic_{r}_structural' for r in frequency}
dynamic_gpu_b1=sum(frequency[r]*latency_value(dynamic_names[r],'batch1') for r in frequency)
dynamic_gpu_b32=sum(frequency[r]*latency_value(dynamic_names[r],'batch32') for r in frequency)
dynamic_params_active=sum(frequency[r]*parameter_counts[dynamic_names[r]] for r in frequency)
dynamic_flops=sum(frequency[r]*flop_counts[dynamic_names[r]] for r in frequency)
dynamic_storage=sum(checkpoint_bytes[dynamic_names[r]] for r in frequency)
mask_storage=sum((SEED_PRUNING_DIR/f'{r}_joint_keep_mask.csv').stat().st_size for r in frequency)

deployment_rows=[
 {'scenario':'unpruned_baseline','stored_models':1,'active_parameters':base_params,'total_stored_checkpoint_bytes':checkpoint_bytes['baseline'],'flops_per_sample':base_flops,'gpu_model_ms_batch1':latency_value('baseline','batch1'),'gpu_model_ms_batch32':latency_value('baseline','batch32'),'detector_ms_per_window':0.0},
 {'scenario':'soft_mask_dynamic_single_dense_model','stored_models':1,'active_parameters':base_params,'total_stored_checkpoint_bytes':checkpoint_bytes['baseline']+mask_storage,'flops_per_sample':base_flops,'gpu_model_ms_batch1':latency_value('baseline','batch1'),'gpu_model_ms_batch32':latency_value('baseline','batch32'),'detector_ms_per_window':stl_mean},
 {'scenario':'static_structural','stored_models':1,'active_parameters':parameter_counts['static_structural'],'total_stored_checkpoint_bytes':checkpoint_bytes['static_structural'],'flops_per_sample':flop_counts['static_structural'],'gpu_model_ms_batch1':latency_value('static_structural','batch1'),'gpu_model_ms_batch32':latency_value('static_structural','batch32'),'detector_ms_per_window':0.0},
 {'scenario':'dynamic_structural_three_models','stored_models':3,'active_parameters':dynamic_params_active,'total_stored_checkpoint_bytes':dynamic_storage,'flops_per_sample':dynamic_flops,'gpu_model_ms_batch1':dynamic_gpu_b1,'gpu_model_ms_batch32':dynamic_gpu_b32,'detector_ms_per_window':stl_mean},
]
deployment_df=pd.DataFrame(deployment_rows); deployment_df['total_e2e_ms_per_window_sequential']=deployment_df.gpu_model_ms_batch1+deployment_df.detector_ms_per_window
deployment_df['active_parameters_vs_baseline_percent']=(deployment_df.active_parameters/base_params-1)*100
deployment_df['flops_vs_baseline_percent']=(deployment_df.flops_per_sample/base_flops-1)*100
deployment_df['stored_mib']=deployment_df.total_stored_checkpoint_bytes/1024**2
for count in [1000,1000000]:
    deployment_df[f'model_only_batched_seconds_{count}']=deployment_df.gpu_model_ms_batch32*np.ceil(count/32)/1000
    deployment_df[f'e2e_sequential_seconds_{count}']=deployment_df.total_e2e_ms_per_window_sequential*count/1000
display(deployment_df)
print(f'STL: {stl_mean:.3f} ± {stl_std:.3f} ms/window over {STL_WINDOWS} windows')

In [ ]:
# 8) KAYDET VE SON KONTROL
correctness_df.to_csv(EFFICIENCY_DIR/'structural_correctness.csv',index=False)
five_seed_storage_df.to_csv(EFFICIENCY_DIR/'baseline_checkpoint_storage_5seeds.csv',index=False)
model_metrics_df.to_csv(EFFICIENCY_DIR/'model_parameters_flops_storage.csv',index=False)
latency_raw_df.to_csv(EFFICIENCY_DIR/'randomized_interleaved_latency_raw.csv',index=False)
latency_summary_df.to_csv(EFFICIENCY_DIR/'randomized_interleaved_latency_summary.csv',index=False)
pd.DataFrame({'latency_ms':stl_times}).to_csv(EFFICIENCY_DIR/'stl_latency_raw.csv',index=False)
pd.DataFrame([{'mean_ms':stl_mean,'std_ms':stl_std,'windows':STL_WINDOWS}]).to_csv(EFFICIENCY_DIR/'stl_latency_summary.csv',index=False)
deployment_df.to_csv(EFFICIENCY_DIR/'honest_deployment_summary.csv',index=False)
metadata={'representative_seed':REPRESENTATIVE_SEED,'tslib_commit':TSLIB_COMMIT,'device':str(device),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'gpu_repeats':GPU_REPEATS,'cpu_repeats':CPU_REPEATS,'stl_windows':STL_WINDOWS}
(EFFICIENCY_DIR/'benchmark_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('Kaydedildi:',EFFICIENCY_DIR)


In [ ]:
# 9) BEŞ SEED İÇİN MASKELERİN KATMAN DAĞILIMI

rows = []

for seed in [7, 42, 1234, 2026, 3407]:
    seed_dir = PRUNING_DIR / f"seed_{seed}"

    mask_files = {
        "static": seed_dir / "static_keep_mask.csv",
        "dynamic_trend": seed_dir / "trend_joint_keep_mask.csv",
        "dynamic_seasonal": seed_dir / "seasonal_joint_keep_mask.csv",
        "dynamic_residual": seed_dir / "residual_joint_keep_mask.csv",
    }

    for method, path in mask_files.items():
        mask = pd.read_csv(path).to_numpy(dtype=float)

        active_per_layer = mask.sum(axis=1).astype(int)

        assert mask.shape == (3, 8)
        assert active_per_layer.sum() == 18

        rows.append({
            "seed": seed,
            "method": method,
            "layer_0_active_heads": active_per_layer[0],
            "layer_1_active_heads": active_per_layer[1],
            "layer_2_active_heads": active_per_layer[2],
            "total_active_heads": active_per_layer.sum(),
        })

mask_distribution_df = pd.DataFrame(rows)

display(mask_distribution_df)

mask_distribution_df.to_csv(
    EFFICIENCY_DIR / "mask_layer_distribution_5seeds.csv",
    index=False,
)

print(
    "Kaydedildi:",
    EFFICIENCY_DIR / "mask_layer_distribution_5seeds.csv",
)

In [ ]:
# 10) BEŞ SEED İÇİN MASK-SHAPE LATENCY SENSITIVITY

SENSITIVITY_REPEATS = 30
SENSITIVITY_WARMUP = 5

sensitivity_models = {}

for seed in [7, 42, 1234, 2026, 3407]:
    seed_dir = PRUNING_DIR / f"seed_{seed}"

    mask_paths = {
        "static": seed_dir / "static_keep_mask.csv",
        "dynamic_trend": seed_dir / "trend_joint_keep_mask.csv",
        "dynamic_seasonal": seed_dir / "seasonal_joint_keep_mask.csv",
        "dynamic_residual": seed_dir / "residual_joint_keep_mask.csv",
    }

    for method, path in mask_paths.items():
        mask = load_mask(path)

        model_name = f"{seed}__{method}"

        # Ağırlık olarak seed=2026 kullanılır.
        # Burada ölçülen değişken yalnızca structural tensor şeklidir.
        sensitivity_models[model_name] = build_structural(mask)

print("Oluşturulan structural model sayısı:", len(sensitivity_models))

sensitivity_raw, sensitivity_summary = interleaved_benchmark(
    model_map=sensitivity_models,
    inputs=sample_gpu,
    target_device=device,
    warmup=SENSITIVITY_WARMUP,
    repeats=SENSITIVITY_REPEATS,
    seed=BENCHMARK_SEED + 10,
    batch_label="batch32_mask_shape_sensitivity",
)

parsed_rows = []

for _, row in sensitivity_summary.iterrows():
    seed_text, method = row["setting"].split("__", 1)

    parsed_rows.append({
        "seed": int(seed_text),
        "method": method,
        "mean_ms_batch32": row["mean_ms"],
        "std_ms_batch32": row["std_ms"],
        "p50_ms_batch32": row["p50_ms"],
        "p95_ms_batch32": row["p95_ms"],
    })

mask_shape_latency_df = pd.DataFrame(parsed_rows)

display(
    mask_shape_latency_df.sort_values(
        ["seed", "method"]
    )
)

In [ ]:
# 11) STATIC VE DYNAMIC BEKLENEN LATENCY ÖZETİ

dynamic_expected_rows = []

for seed in [7, 42, 1234, 2026, 3407]:
    seed_rows = mask_shape_latency_df[
        mask_shape_latency_df["seed"] == seed
    ].set_index("method")

    expected_latency = (
        frequency["trend"]
        * seed_rows.loc["dynamic_trend", "mean_ms_batch32"]
        + frequency["seasonal"]
        * seed_rows.loc["dynamic_seasonal", "mean_ms_batch32"]
        + frequency["residual"]
        * seed_rows.loc["dynamic_residual", "mean_ms_batch32"]
    )

    dynamic_expected_rows.append({
        "seed": seed,
        "method": "dynamic_expected",
        "mean_ms_batch32": expected_latency,
    })

dynamic_expected_df = pd.DataFrame(dynamic_expected_rows)

static_latency_df = (
    mask_shape_latency_df[
        mask_shape_latency_df["method"] == "static"
    ][["seed", "method", "mean_ms_batch32"]]
)

latency_seed_comparison = pd.concat(
    [static_latency_df, dynamic_expected_df],
    ignore_index=True,
)

display(latency_seed_comparison)

latency_mean_std = (
    latency_seed_comparison
    .groupby("method")
    .agg(
        latency_mean_ms=("mean_ms_batch32", "mean"),
        latency_std_ms=("mean_ms_batch32", "std"),
        min_ms=("mean_ms_batch32", "min"),
        max_ms=("mean_ms_batch32", "max"),
    )
    .reset_index()
)

display(latency_mean_std)

mask_shape_latency_df.to_csv(
    EFFICIENCY_DIR / "mask_shape_latency_5seeds.csv",
    index=False,
)

latency_seed_comparison.to_csv(
    EFFICIENCY_DIR / "static_dynamic_latency_5seeds.csv",
    index=False,
)

latency_mean_std.to_csv(
    EFFICIENCY_DIR / "static_dynamic_latency_mean_std.csv",
    index=False,
)

del sensitivity_models
torch.cuda.empty_cache()
gc.collect()

print("Beş-seed mask-shape latency sensitivity tamamlandı.")